# Method 2: Direct Network Access to DGX Server

This notebook verifies direct network connectivity from this machine to the DGX server over the local network or Tailscale VPN on port `30001` without requiring an SSH tunnel.

**Configuration:**
- **DGX Host IP:** `100.85.128.70`
- **Port:** `30001`
- **Base URL:** `http://100.85.128.70:30001/v1`
- **API Key:** `EMPTY`

### Step 1: Pre-Flight TCP Socket Reachability Test

In [ ]:
import socket

DGX_IP = "100.85.128.70"
DGX_PORT = 30001

print(f"Probing TCP connection to {DGX_IP}:{DGX_PORT}...")
s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
s.settimeout(3.0)
try:
    s.connect((DGX_IP, DGX_PORT))
    s.close()
    print(f"✅ Direct TCP connection to {DGX_IP}:{DGX_PORT} succeeded!")
except Exception as e:
    print(f"❌ Connection failed: {e}")
    print("💡 Ensure your VPN/Tailscale is running and port 30001 is open on DGX firewall.")

### Step 2: Initialize OpenAI Client with Direct Network Access

In [ ]:
from openai import OpenAI

# Direct Network Access to DGX
client = OpenAI(
    base_url="http://100.85.128.70:30001/v1",
    api_key="EMPTY"
)

print("OpenAI client configured for DGX Direct Network Access.")

### Step 3: Fetch Available Models on DGX

In [ ]:
import time

t0 = time.time()
models = client.models.list()
latency = time.time() - t0

print(f"✅ Fetched models in {latency:.2f}s:")
for m in models.data:
    print(f"  • Model ID: {m.id} (Owner: {getattr(m, 'owned_by', 'unknown')})")

active_model = models.data[0].id if models.data else None
print(f"\nActive target model: {active_model}")

### Step 4: Run a Test Chat Completion

In [ ]:
if active_model:
    print(f"Sending test prompt to {active_model}...")
    t0 = time.time()
    response = client.chat.completions.create(
        model=active_model,
        messages=[
            {"role": "user", "content": "Hello from APEX v2! Please reply with 'Direct DGX connection confirmed!' in 10 words or less."}
        ],
        max_tokens=60,
        temperature=0.0
    )
    elapsed = time.time() - t0
    
    print(f"\n✅ Response received in {elapsed:.2f}s:")
    print("-" * 50)
    print(response.choices[0].message.content.strip())
    print("-" * 50)
    if response.usage:
        print(f"Prompt tokens: {response.usage.prompt_tokens}")
        print(f"Completion tokens: {response.usage.completion_tokens}")
        print(f"Total tokens: {response.usage.total_tokens}")

### Step 5: Test Streaming Response (Low-Latency Verification)

In [ ]:
if active_model:
    print("Testing token streaming:")
    stream = client.chat.completions.create(
        model=active_model,
        messages=[
            {"role": "user", "content": "Count from 1 to 5."}
        ],
        max_tokens=30,
        stream=True,
        temperature=0.0
    )
    for chunk in stream:
        content = chunk.choices[0].delta.content or ""
        print(content, end="", flush=True)
    print("\n\n✅ Streaming test complete!")